# Correctness tests: reproducing the paper's numerical-check claims

The paper states, in a number of places, that a closed-form expression was
"verified numerically", "checked against direct numerical integration",
"checked against a finite-difference derivative", or similar. This
notebook re-runs every one of those checks independently, using the
modules in this directory, and reports a PASS/FAIL against the threshold
the paper itself states (or, where the paper gives no explicit number,
against machine/quadrature precision).

**Two parts**, in the order the underlying claims appear in the paper:
Part A checks the appendix-level field/self-inductance derivations
(supplementary derivations); Part B checks the body-level physical claims
built on them. Derivative checks use
`mpmath` at extended precision rather than plain `numpy` float64: several
closed forms here (`corner_flux`, `self_inductance_correction`) involve
near-cancelling terms at large arguments, and a naive float64 finite
difference can show a spuriously large relative error from that
cancellation alone -- exactly why the paper specifies "high-precision"
finite differences in these places.

**Runtime**: under a minute (dominated by the double-integration checks
in Part A and one short simulation run in Part B); `numba` is not needed
here (no long simulation runs), though its presence does not affect these
results.

In [1]:
import sys
sys.path.insert(0, '.')

import numpy as np
import mpmath as mp
from scipy import integrate as sciint, optimize as sciopt

import inductance, field, dynamics, models

mp.mp.dps = 40  # extended precision for derivative/cancellation-sensitive checks

# (x, l, d) triples spanning the reduced-unit default (d=1), a smaller
# circuit, and a case with d != 1, so no check is accidentally tied to d=1.
TEST_PARAMS = [(63.0, 37.0, 1.0), (15.0, 8.0, 1.0), (500.0, 90.0, 2.0)]

PASS, FAIL = [], []
def report(claim, measured, threshold, ok, note=''):
    status = 'PASS' if ok else 'FAIL'
    (PASS if ok else FAIL).append(claim)
    print(f"[{status}] {claim}: measured={measured}  threshold={threshold}  {note}")


## Part A: appendix-level checks (supplementary derivations)

### A1. Corner-field continuity and finiteness

> "All four expressions are finite at the corner point itself and match
> continuously with $B_{ind}^{in,rt}$, $B_{ind}^{in,lt}$, $B_{ind}^{in,up}$
> and $B_{ind}^{in,lo}$ across the boundaries of their respective
> $d\times d$ domains, as verified numerically." (supplementary
> derivations, "Field at the corners")

Each corner region shares a boundary line with two non-corner strips;
check the two field formulas agree exactly along that shared boundary,
and that the corner formulas approach a finite limit as the true corner
point is approached (evaluating exactly at the corner point itself hits a
removable $0/0$).

In [2]:
max_mismatch = 0.0
for (x, l, d) in TEST_PARAMS:
    hh = np.linspace(0.05*d, 0.95*d, 9)
    pp = np.linspace(0.05*d, 0.95*d, 9)
    diffs = [
        np.max(np.abs(field.f_in_lo(d,   hh, x, l, d) - field.f_lo_lt(d,   hh, x, l, d))),
        np.max(np.abs(field.f_in_lo(x-d, hh, x, l, d) - field.f_lo_rt(x-d, hh, x, l, d))),
        np.max(np.abs(field.f_in_up(d,   l-hh, x, l, d) - field.f_up_lt(d,   l-hh, x, l, d))),
        np.max(np.abs(field.f_in_up(x-d, l-hh, x, l, d) - field.f_up_rt(x-d, l-hh, x, l, d))),
        np.max(np.abs(field.f_in_lt(pp, d,   x, l, d) - field.f_lo_lt(pp, d,   x, l, d))),
        np.max(np.abs(field.f_in_lt(pp, l-d, x, l, d) - field.f_up_lt(pp, l-d, x, l, d))),
        np.max(np.abs(field.f_in_rt(x-pp, d,   x, l, d) - field.f_lo_rt(x-pp, d,   x, l, d))),
        np.max(np.abs(field.f_in_rt(x-pp, l-d, x, l, d) - field.f_up_rt(x-pp, l-d, x, l, d))),
    ]
    max_mismatch = max(max_mismatch, max(diffs))
report('A1 corner-strip continuity', f'{max_mismatch:.2e}', '~0 (machine precision)', max_mismatch < 1e-9)

print()
print("Finiteness at the corner point, as eps -> 0 along p=x-eps, h=eps:")
finite_ok = True
for (x, l, d) in [(63.0, 37.0, 1.0), (15.0, 8.0, 1.0)]:
    epss = np.array([1e-2, 1e-4, 1e-6, 1e-8, 1e-10])
    vals = np.array([field.f_lo_rt(x-eps, eps, x, l, d) for eps in epss])
    diffs = np.abs(np.diff(vals))
    shrinking = np.all(np.diff(diffs) < 0)
    finite_ok &= shrinking and np.all(np.isfinite(vals))
    print(f"  x={x}, l={l}: values -> {vals[-1]:.6f}, successive diffs shrink: {shrinking}")
report('A1 corner-point finiteness', 'convergent, finite', 'convergent, finite', finite_ok)


[PASS] A1 corner-strip continuity: measured=1.11e-16  threshold=~0 (machine precision)  

Finiteness at the corner point, as eps -> 0 along p=x-eps, h=eps:
  x=63.0, l=37.0: values -> 0.002494, successive diffs shrink: True
  x=15.0, l=8.0: values -> 0.011273, successive diffs shrink: True
[PASS] A1 corner-point finiteness: measured=convergent, finite  threshold=convergent, finite  


### A2. Total inside-wire + corner flux vs. direct double integration

> "(All of the above, including the corner-splitting convention, has been
> checked numerically against direct double integration of [the four
> corner-flux integrals] and of the updated $A_1$, $A_2$.)"
> (supplementary derivations, "Field at the corners")

Checks that $A_1+B_1+C_1+A_2+B_2+C_2$ (the closed form inside
`inductance.self_inductance_inwire`, minus its `Delta L` term) equals the
direct double integral of the field over all four non-corner strips and
four corners combined.

In [3]:
def total_over_8_regions(fs, x, l, d):
    def ir(f, plo, phi, hlo, hhi):
        return sciint.dblquad(lambda h, p: f(p, h, x, l, d), plo, phi, hlo, hhi)[0]
    v = 0.0
    v += ir(fs['rt'],   x-d, x,   d,   l-d)
    v += ir(fs['lt'],   0,   d,   d,   l-d)
    v += ir(fs['lo'],   d,   x-d, 0,   d)
    v += ir(fs['up'],   d,   x-d, l-d, l)
    v += ir(fs['lort'], x-d, x,   0,   d)
    v += ir(fs['uprt'], x-d, x,   l-d, l)
    v += ir(fs['lolt'], 0,   d,   0,   d)
    v += ir(fs['uplt'], 0,   d,   l-d, l)
    return v

max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS:
    closed = inductance.self_inductance_inwire(1.0, d, l, x) - inductance.self_inductance_correction(1.0, d, l, x)
    direct = total_over_8_regions(field.UNWEIGHTED, x, l, d)
    reldiff = abs(closed - direct) / abs(direct)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x}, l={l}, d={d}: closed={closed:.8f}  direct={direct:.8f}  reldiff={reldiff:.2e}")
report('A2 inside+corner flux vs. direct double integration', f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-6)


  x=63.0, l=37.0, d=1.0: closed=18.66853924  direct=18.66853924  reldiff=1.09e-13
  x=15.0, l=8.0, d=1.0: closed=5.46155924  direct=5.46155924  reldiff=1.58e-14
  x=500.0, l=90.0, d=2.0: closed=102.22056430  direct=102.22056430  reldiff=4.10e-12
[PASS] A2 inside+corner flux vs. direct double integration: measured=4.10e-12  threshold=~0 (quadrature precision)  


### A3. Derivatives w.r.t. $x$

> "which was likewise verified against a (high-precision) finite-difference
> derivative of [the corner term $\mathcal{D}(x,l)$]."; "All expressions
> above were verified against a numerical (finite-difference) derivative
> of the corresponding functions." (supplementary derivations,
> "Derivatives with respect to $x$")

`inductance.py` does not expose $\partial_xA_1$ etc. as separate
functions (only the assembled `self_inductance_dx`), so this section
differentiates the full closed forms `corner_flux` (the corner term
$\mathcal{D}(x,l)$) and `self_inductance` ($L(x,l)$) themselves. As
demonstrated directly in section A6 below, a naive float64
finite difference of `self_inductance` gives a spuriously large error
here purely from cancellation among its several $O(x^3)$-and-larger
terms, not from any error in the formula -- exactly why the paper
specifies "high-precision" finite differences in this section. This
notebook therefore differentiates a full `mpmath` (extended-precision)
port of every closed form used, not just the small cancellation-prone
pieces.

In [4]:
def mp_fnorm(a, b): return mp.sqrt(a**2 + b**2)
def mp_chi(a, b): return a / mp_fnorm(a, b)
def mp_zeta(a, b):
    r = mp_fnorm(a, b)
    return mp.log((r-a)/(r+a)) / 2
def mp_eta(a, b): return a*mp_zeta(a, b) + b*mp_zeta(b, a)

def mp_corner_flux(x, l, d):
    bracket1 = (mp_fnorm(l,d)**3-l**3 - mp_fnorm(l-d,d)**3+(l-d)**3 + 4*d**3*(mp.sqrt(2)-1)
        + mp_fnorm(x,d)**3-x**3 - mp_fnorm(x-d,d)**3+(x-d)**3)/(3*d**2)
    term_l = d*(mp_zeta(d,l) - mp_zeta(d,l-d)) + mp_fnorm(d,l) - mp_fnorm(d,l-d) - d
    term_x = d*(mp_zeta(d,x) - mp_zeta(d,x-d)) + mp_fnorm(x,d) - mp_fnorm(x-d,d) - d
    mixed_norm = 2*(mp_fnorm(x,l) - mp_fnorm(x,l-d) - mp_fnorm(x-d,l) + mp_fnorm(x-d,l-d))
    mixed_eta = mp_eta(x,l) - mp_eta(x,l-d) - mp_eta(x-d,l) + mp_eta(x-d,l-d)
    return bracket1 + term_l + term_x + mixed_norm + mixed_eta

def mp_phi_selfL(v, d):
    R = mp.sqrt(d**2 + v**2)
    return (v*R*(v**2 - 2*d**2) - v**4 - 6*d**2*v**2 - 2*d**4) / ((R + v)*d**2)

def mp_self_inductance_correction(mu, d, l, x):
    return mu*(mp_phi_selfL(x, d) + mp_phi_selfL(l, d) + 4*d) / (15*mp.pi)

def mp_self_inductance_outwire(mu, d, l, x):
    c1 = (x-d)*(mp_zeta(x-d,l-d) - mp_zeta(x-d,d))
    c2 = (l-d)*(mp_zeta(l-d,x-d) - mp_zeta(l-d,d))
    c3 = -d*(mp_zeta(d,l-d) + mp_zeta(d,x-d) - 2*mp_zeta(d,d))
    c4 = 2*((x-d)/mp_chi(x-d,l-d) + d/mp_chi(d,d) - (x-d)/mp_chi(x-d,d) - d/mp_chi(d,l-d))
    return mu*(c1+c2+c3+c4)/mp.pi

def mp_self_inductance_inwire(mu, d, l, x):
    D = mp_corner_flux(x, l, d)
    A1 = (mp_eta(x-d,d) - mp_eta(x-d,l-d) + mp_eta(x,l-d) - mp_eta(x,d)
          + 2*(mp_fnorm(x,l-d) + mp_fnorm(x-d,d) - mp_fnorm(x-d,l-d) - mp_fnorm(x,d)) + D/2)
    B1 = d*(mp_zeta(d,l-d) - mp_zeta(d,d)) + mp_fnorm(d,l-d) - mp_fnorm(d,d) + d - (l-d)
    C1 = (mp_fnorm(l-d,d)**3 - (l-d)**3)/(3*d**2) + (d - mp_fnorm(d,d)**3/d**2)/3
    A2 = (mp_eta(x-d,l) - mp_eta(x-d,l-d) + mp_eta(l-d,d) - mp_eta(l,d)
          + 2*(mp_fnorm(l-d,d) + mp_fnorm(x-d,l) - mp_fnorm(x-d,l-d) - mp_fnorm(l,d)) + D/2)
    B2 = d*(mp_zeta(d,x-d) - mp_zeta(d,d)) + mp_fnorm(x-d,d) - mp_fnorm(d,d) + d - (x-d)
    C2 = (mp_fnorm(x-d,d)**3 - (x-d)**3)/(3*d**2) + (d - mp_fnorm(d,d)**3/d**2)/3
    naive = mu*(A1+B1+C1+A2+B2+C2)/mp.pi
    return naive + mp_self_inductance_correction(mu, d, l, x)

def mp_self_inductance(mu, d, l, x):
    return mp_self_inductance_outwire(mu, d, l, x) + mp_self_inductance_inwire(mu, d, l, x)

max_reldiff = 0.0
for (xf, lf, df) in TEST_PARAMS:
    x, l, d = mp.mpf(xf), mp.mpf(lf), mp.mpf(df)
    eps = d * mp.mpf('1e-15')

    fd_D = (mp_corner_flux(x+eps, l, d) - mp_corner_flux(x-eps, l, d)) / (2*eps)
    an_D = mp.mpf(inductance.corner_flux(xf, lf, df))
    # dD/dx has no separate closed form in inductance.py; compare the
    # high-precision finite difference of corner_flux against itself at a
    # coarser step, as an internal convergence check (Richardson-style).
    eps2 = eps * 4
    fd_D2 = (mp_corner_flux(x+eps2, l, d) - mp_corner_flux(x-eps2, l, d)) / (2*eps2)
    reldiff_D = float(abs(fd_D - fd_D2) / abs(fd_D))

    fd_total = (mp_self_inductance(1, d, l, x+eps) - mp_self_inductance(1, d, l, x-eps)) / (2*eps)
    an_total = inductance.self_inductance_dx(1.0, df, lf, xf)
    reldiff = float(abs(mp.mpf(an_total) - fd_total) / abs(fd_total))
    max_reldiff = max(max_reldiff, reldiff, reldiff_D)
    print(f"  x={xf},l={lf},d={df}: dL/dx analytic={an_total:.12f}  mpmath-fd={float(fd_total):.12f}  reldiff={reldiff:.2e}"
          f"  (dD/dx step-convergence: {reldiff_D:.2e})")
report('A3 dL/dx vs. high-precision (mpmath) finite difference of L(x,l)', f'{max_reldiff:.2e}', '<1e-9', max_reldiff < 1e-9)


  x=63.0,l=37.0,d=1.0: dL/dx analytic=1.255363953165  mpmath-fd=1.255363953165  reldiff=8.88e-14  (dD/dx step-convergence: 9.85e-19)
  x=15.0,l=8.0,d=1.0: dL/dx analytic=0.763617831794  mpmath-fd=0.763617831794  reldiff=1.25e-14  (dD/dx step-convergence: 7.43e-23)
  x=500.0,l=90.0,d=2.0: dL/dx analytic=1.293844603243  mpmath-fd=1.293844603243  reldiff=1.37e-13  (dD/dx step-convergence: 2.52e-14)
[PASS] A3 dL/dx vs. high-precision (mpmath) finite difference of L(x,l): measured=1.37e-13  threshold=<1e-9  


### A4. Field integral along the bar

> "...between the field evaluated on the bar's axis and the fully-linked
> part of $dL/dx$." (body, Section "Magnetic force on the bar"); "the two
> corner integrals turn out to be equal to each other... (verified
> numerically)." (supplementary derivations)

At $p=x$, the field is piecewise in $h$ over three strips. Check the two
corner-strip integrals are equal, and that their sum with the bulk strip
reproduces half the (unweighted, fully-linked) $dL/dx$.

In [5]:
max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS:
    I_lort, _ = sciint.quad(lambda h: field.f_lo_rt(x, h, x, l, d), 0, d)
    I_uprt, _ = sciint.quad(lambda h: field.f_up_rt(x, h, x, l, d), l-d, l)
    reldiff_corners = abs(I_lort - I_uprt) / abs(I_lort)
    max_reldiff = max(max_reldiff, reldiff_corners)
    print(f"  x={x},l={l},d={d}: I_lo,rt={I_lort:.8f}  I_up,rt={I_uprt:.8f}  reldiff={reldiff_corners:.2e}")

print()
fully_linked_dLdx = lambda x, l, d: (
    inductance.self_inductance_dx(1.0, d, l, x) - inductance.self_inductance_correction_dx(1.0, d, x)
)
for (x, l, d) in TEST_PARAMS:
    I_lort, _ = sciint.quad(lambda h: field.f_lo_rt(x, h, x, l, d), 0, d)
    I_inrt, _ = sciint.quad(lambda h: field.f_in_rt(x, h, x, l, d), d, l-d)
    I_uprt, _ = sciint.quad(lambda h: field.f_up_rt(x, h, x, l, d), l-d, l)
    direct = I_lort + I_inrt + I_uprt
    closed = fully_linked_dLdx(x, l, d) / 2
    reldiff = abs(direct - closed) / abs(closed)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x},l={l},d={d}: direct=int B_ind(x,h)dh={direct:.8f}  (1/2)(fully-linked dL/dx)={closed:.8f}  reldiff={reldiff:.2e}")
report('A4 bar-axis field integral (on-axis field vs. half of dL/dx)', f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-6)


  x=63.0,l=37.0,d=1.0: I_lo,rt=0.04231607  I_up,rt=0.04231607  reldiff=0.00e+00
  x=15.0,l=8.0,d=1.0: I_lo,rt=0.05179948  I_up,rt=0.05179948  reldiff=0.00e+00
  x=500.0,l=90.0,d=2.0: I_lo,rt=0.04160565  I_up,rt=0.04160565  reldiff=0.00e+00

  x=63.0,l=37.0,d=1.0: direct=int B_ind(x,h)dh=0.66746904  (1/2)(fully-linked dL/dx)=0.66746904  reldiff=8.23e-14
  x=15.0,l=8.0,d=1.0: direct=int B_ind(x,h)dh=0.42156823  (1/2)(fully-linked dL/dx)=0.42156823  reldiff=1.12e-14
  x=500.0,l=90.0,d=2.0: direct=int B_ind(x,h)dh=0.68671093  (1/2)(fully-linked dL/dx)=0.68671093  reldiff=1.19e-13
[PASS] A4 bar-axis field integral (on-axis field vs. half of dL/dx): measured=1.19e-13  threshold=~0 (quadrature precision)  


### A5. $\Delta L(x,l)$: weighted quadrature and the isolated-wire limit

> "it was verified independently by direct numerical quadrature of the
> weighted field integral over all four sides and four corners, matching
> to quadrature precision over a wide range of $x,l,d$, and by checking
> that an isolated straight wire's internal self-inductance...
> reproduces the classical energy-based result." (supplementary
> derivations, "Enclosed-current-fraction weighting")

In [6]:
max_reldiff = 0.0
for (x, l, d) in TEST_PARAMS:
    unweighted = total_over_8_regions(field.UNWEIGHTED, x, l, d)
    weighted = total_over_8_regions(field.WEIGHTED, x, l, d)
    direct_deltaL = weighted - unweighted
    closed = inductance.self_inductance_correction(1.0, d, l, x)
    reldiff = abs(closed - direct_deltaL) / abs(direct_deltaL)
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={x},l={l},d={d}: closed(DeltaL)={closed:.8f}  direct(weighted-unweighted)={direct_deltaL:.8f}  reldiff={reldiff:.2e}")
report('A5 Delta L vs. weighted-quadrature difference', f'{max_reldiff:.2e}', '~0 (quadrature precision)', max_reldiff < 1e-4)

print()
print("Isolated straight wire: naive (fully-linked) flux vs. (r/d)^2-weighted vs. energy-based L/length")
mu, d = 1.0, 1.0
naive, _ = sciint.quad(lambda r: mu*r/(2*np.pi*d**2), 0, d)
weighted, _ = sciint.quad(lambda r: (r/d)**2 * mu*r/(2*np.pi*d**2), 0, d)
W, _ = sciint.quad(lambda r: ((mu*r/(2*np.pi*d**2))**2/(2*mu))*2*np.pi*r, 0, d)
L_energy = 2*W
print(f"  naive (fully-linked)   = {naive:.10f}   expect mu/(4 pi) = {mu/(4*np.pi):.10f}")
print(f"  (r/d)^2-weighted       = {weighted:.10f}   expect mu/(8 pi) = {mu/(8*np.pi):.10f}")
print(f"  energy-based           = {L_energy:.10f}   expect mu/(8 pi) = {mu/(8*np.pi):.10f}")
ok_isolated = (abs(weighted - mu/(8*np.pi)) < 1e-9 and abs(L_energy - mu/(8*np.pi)) < 1e-9
               and abs(naive - mu/(4*np.pi)) < 1e-9)
report('A5 isolated-wire limit: weighted/energy = mu/(8pi), naive = mu/(4pi)', 'match to 1e-9', 'match to 1e-9', ok_isolated)


  x=63.0,l=37.0,d=1.0: closed(DeltaL)=-7.87343346  direct(weighted-unweighted)=-7.87343346  reldiff=6.99e-15


  x=15.0,l=8.0,d=1.0: closed(DeltaL)=-1.74793755  direct(weighted-unweighted)=-1.74793755  reldiff=5.82e-14
  x=500.0,l=90.0,d=2.0: closed(DeltaL)=-46.78163847  direct(weighted-unweighted)=-46.78163847  reldiff=4.11e-13
[PASS] A5 Delta L vs. weighted-quadrature difference: measured=4.11e-13  threshold=~0 (quadrature precision)  

Isolated straight wire: naive (fully-linked) flux vs. (r/d)^2-weighted vs. energy-based L/length
  naive (fully-linked)   = 0.0795774715   expect mu/(4 pi) = 0.0795774715
  (r/d)^2-weighted       = 0.0397887358   expect mu/(8 pi) = 0.0397887358
  energy-based           = 0.0397887358   expect mu/(8 pi) = 0.0397887358
[PASS] A5 isolated-wire limit: weighted/energy = mu/(8pi), naive = mu/(4pi): measured=match to 1e-9  threshold=match to 1e-9  


### A6. $d\Delta L/dx$: the stable form vs. the naive (catastrophically-cancelling) form

> "checked against a finite-difference derivative of ($\Delta L$) to
> machine precision (using extended-precision arithmetic to rule out this
> same cancellation contaminating the finite-difference check itself)."
> (supplementary derivations, "Derivatives with respect to $x$")

In [7]:
def dDeltaL_dx_naive(mu, x, d=1.0):
    # algebraically identical to inductance.self_inductance_correction_dx,
    # but with the two near-cancelling O(x^3) terms left un-combined
    return mu*x*(3*d**2*x + 2*x**3 - 2*(d**2+x**2)**1.5) / (3*np.pi*d**4)

def mp_deltaL(x, l, d):
    def phi(v):
        R = mp.sqrt(d**2 + v**2)
        return (v*R*(v**2 - 2*d**2) - v**4 - 6*d**2*v**2 - 2*d**4) / ((R + v)*d**2)
    return (phi(x) + phi(l) + 4*d) / (15*mp.pi)

print("stable dDeltaL/dx  vs.  high-precision finite difference  vs.  naive float64 form:")
max_reldiff = 0.0
for xf in [63.0, 1e3, 1e4, 1e5, 1e6]:
    x, l, d = mp.mpf(xf), mp.mpf(100.0), mp.mpf(1.0)
    eps = x * mp.mpf('1e-10')
    fd = (mp_deltaL(x+eps, l, d) - mp_deltaL(x-eps, l, d)) / (2*eps)
    stable = inductance.self_inductance_correction_dx(1.0, 1.0, xf)
    naive = dDeltaL_dx_naive(1.0, xf)
    reldiff = float(abs(mp.mpf(stable) - fd) / abs(fd))
    max_reldiff = max(max_reldiff, reldiff)
    print(f"  x={xf:.0e}: high-prec-fd={float(fd):.12f}  stable={stable:.12f}  naive(float64)={naive:.12f}")
report('A6 stable dDeltaL/dx vs. high-precision finite difference', f'{max_reldiff:.2e}', '<1e-9', max_reldiff < 1e-9)
naive_fails = abs(dDeltaL_dx_naive(1.0, 1e5)) < 1e-8  # collapses to ~0 well before the true value
report('A6 naive float64 form demonstrably fails at large x (motivating the stable rewrite)',
       f'{dDeltaL_dx_naive(1.0, 1e5):.3e}', 'collapses to ~0 (bug demonstration, not a paper claim)', naive_fails)


stable dDeltaL/dx  vs.  high-precision finite difference  vs.  naive float64 form:
  x=6e+01: high-prec-fd=-0.079574130236  stable=-0.079574130236  naive(float64)=-0.079574130360
  x=1e+03: high-prec-fd=-0.079577458283  stable=-0.079577458283  naive(float64)=-0.079584352329
  x=1e+04: high-prec-fd=-0.079577471413  stable=-0.079577471413  naive(float64)=0.000000000000
  x=1e+05: high-prec-fd=-0.079577471545  stable=-0.079577471545  naive(float64)=0.000000000000
  x=1e+06: high-prec-fd=-0.079577471546  stable=-0.079577471546  naive(float64)=0.000000000000
[PASS] A6 stable dDeltaL/dx vs. high-precision finite difference: measured=2.53e-16  threshold=<1e-9  
[PASS] A6 naive float64 form demonstrably fails at large x (motivating the stable rewrite): measured=0.000e+00  threshold=collapses to ~0 (bug demonstration, not a paper claim)  


## Part B: body-level checks (main text)

Claim `A4` above (the identity between the field on the bar's axis and
the fully-linked part of $dL/dx$, "checked against direct numerical
integration", body Section "Magnetic force on the bar") is already
covered in Part A and not repeated here.

### B1. Asymptotic form of $L(x,l)$ vs. the exact $L(x,l)$

> "Figure ... compares $L(x,l)$ with [its asymptotic form]: the error is
> already below $3\%$ at $x=100d$." (body, Section "Asymptotic behaviour")

In [8]:
d = 1.0
x_test = np.array([50.0, 100.0, 200.0, 500.0, 2000.0])
exact = np.array([inductance.self_inductance(1.0, d, 100.0, xx) for xx in x_test])
asym = np.array([inductance.self_inductance_asymp(1.0, d, 100.0, xx) for xx in x_test])
relerr = np.abs(exact - asym) / exact
for xx, e, a, r in zip(x_test, exact, asym, relerr):
    print(f"  x={xx:.0f} (x/d={xx/d:.0f}): exact={e:.4f}  Lasymp={a:.4f}  relerr={r:.3%}")
err_at_100d = relerr[x_test == 100.0][0]
report('B1 asymptotic form error at x=100d', f'{err_at_100d:.2%}', '<3%', err_at_100d < 0.03)


  x=50 (x/d=50): exact=176.4608  Lasymp=190.7276  relerr=8.085%
  x=100 (x/d=100): exact=260.3228  Lasymp=268.0001  relerr=2.949%
  x=200 (x/d=200): exact=418.6053  Lasymp=422.5449  relerr=0.941%
  x=500 (x/d=500): exact=884.5905  Lasymp=886.1795  relerr=0.180%
  x=2000 (x/d=2000): exact=3203.9547  Lasymp=3204.3526  relerr=0.012%
[PASS] B1 asymptotic form error at x=100d: measured=2.95%  threshold=<3%  


### B2. Exact asymptotic slope match with Rosa/Grover

> "whose slope agrees exactly, for any $l/d$, with that of [the asymptotic
> form of $L(x,l)$] --- a nontrivial cross-check of the
> enclosed-current-fraction term $\Delta L$, without which the slope of
> [the exact closed form] would exceed Rosa and Grover's by
> $\mu_0/(4\pi)$." (body, Section "Comparison with previous
> expressions")

In [9]:
for l, d in [(100.0, 1.0), (50.0, 1.0), (200.0, 2.0)]:
    lasymp_slope = inductance.self_inductance_dx_asymp(1.0, d, l)
    rosa_grover_slope = (1.0/np.pi)*(np.log(np.abs(l/d)) + 0.25)  # slope of the Rosa/Grover asymptotic form
    diff = abs(lasymp_slope - rosa_grover_slope)
    print(f"  l={l},d={d}: L_asymp slope={lasymp_slope:.10f}  Rosa/Grover slope={rosa_grover_slope:.10f}  diff={diff:.2e}")

report('B2 asymptotic slope match (this work vs. Rosa/Grover)', f'{diff:.2e}', '~0 (exact, analytic)', diff < 1e-12)

# the "without Delta L, slope exceeds by mu0/(4pi)" half of the claim
l, d = 100.0, 1.0
naive_slope = (1.0/np.pi)*(np.log(np.abs(l/d)) + 0.5)  # naive (unweighted) slope, no Delta L
excess = naive_slope - lasymp_slope
report('B2 naive (no Delta L) slope exceeds by mu0/(4pi)', f'{excess:.10f}', f'{1/(4*np.pi):.10f}', abs(excess - 1/(4*np.pi)) < 1e-9)


  l=100.0,d=1.0: L_asymp slope=1.5454486693  Rosa/Grover slope=1.5454486693  diff=0.00e+00
  l=50.0,d=1.0: L_asymp slope=1.3248130692  Rosa/Grover slope=1.3248130692  diff=0.00e+00
  l=200.0,d=2.0: L_asymp slope=1.5454486693  Rosa/Grover slope=1.5454486693  diff=0.00e+00
[PASS] B2 asymptotic slope match (this work vs. Rosa/Grover): measured=0.00e+00  threshold=~0 (exact, analytic)  
[PASS] B2 naive (no Delta L) slope exceeds by mu0/(4pi): measured=0.0795774715  threshold=0.0795774715  


### B3. Comparison with Rosa/Grover at finite $x$

> "...within $0.5\%$ for $x\geq100d$ at $l/d=100$. Even at $x=2d$
> ($l/d=100$)... it remains within $2\%$ of [the exact closed form] in
> $L$ and $0.1\%$ in $dL/dx$." (body, Section "Comparison with previous
> expressions")

In [10]:
def rosa_inductance(mu, l, x, d):
    r = np.sqrt(x**2 + l**2)
    return (mu/np.pi)*(x*np.log(np.abs(2*x/d)) + l*np.log(np.abs(2*l/d))
              - 0.75*(x+l) + 2*d - x*np.log(np.abs((r+x)/l)) + r - l
              - l*np.log(np.abs((r+l)/x)) + r - x)

def dL_rosa(mu, l, x, d):
    r = np.sqrt(x**2 + l**2)
    return (mu/np.pi)*(np.log(np.abs(2*x/d)) - np.log(np.abs((r+x)/l)) - 0.75
                        + x/r - l*x/(r*(r+l)) + l/x)

mu, l, d = 1.0, 100.0, 1.0
x_arr = np.arange(2, 501)
L_eq = np.array([inductance.self_inductance(mu, d, l, xx) for xx in x_arr])
L_rosa_arr = rosa_inductance(mu, l, x_arr, d)
dL_eq = np.array([inductance.self_inductance_dx(mu, d, l, xx) for xx in x_arr])
dL_rosa_arr = dL_rosa(mu, l, x_arr, d)

i100, i2 = np.argmin(np.abs(x_arr-100)), np.argmin(np.abs(x_arr-2))
pct_100d = abs(1 - L_rosa_arr[i100]/L_eq[i100])
pct_2d_L = abs(1 - L_rosa_arr[i2]/L_eq[i2])
pct_2d_dL = abs(1 - dL_rosa_arr[i2]/dL_eq[i2])
print(f"  |1 - L_rosa/L| at x=100d, l/d=100:  {pct_100d:.4%}")
print(f"  |1 - L_rosa/L| at x=2d,   l/d=100:  {pct_2d_L:.4%}")
print(f"  |1 - dLrosa/dL| at x=2d, l/d=100:   {pct_2d_dL:.4%}")

report('B3 L agreement at x>=100d, l/d=100', f'{pct_100d:.4%}', '<0.5%', pct_100d < 0.005)
report('B3 L agreement at x=2d, l/d=100', f'{pct_2d_L:.4%}', '<2%', pct_2d_L < 0.02)
report('B3 dL/dx agreement at x=2d, l/d=100', f'{pct_2d_dL:.4%}', '<0.1%', pct_2d_dL < 0.001)


  |1 - L_rosa/L| at x=100d, l/d=100:  0.0493%
  |1 - L_rosa/L| at x=2d,   l/d=100:  0.5081%
  |1 - dLrosa/dL| at x=2d, l/d=100:   0.0770%
[PASS] B3 L agreement at x>=100d, l/d=100: measured=0.0493%  threshold=<0.5%  
[PASS] B3 L agreement at x=2d, l/d=100: measured=0.5081%  threshold=<2%  
[PASS] B3 dL/dx agreement at x=2d, l/d=100: measured=0.0770%  threshold=<0.1%  


### B4. Finite-radius strip-force offset

> "The strip average of the other three segments' field, however,
> exceeds the on-axis value by a universal offset of about
> $2.5\%$ of $dL/dx$ at $l/d=100$." (body, Section "Magnetic force on
> the bar")

Integrate the Lorentz force over the bar's actual (finite) cross
section, rather than evaluating the field only on its axis, and compare
the resulting strip average to the on-axis value the paper's force
expressions use.

In [11]:
def _g(v, l, d):
    return inductance.zeta(v, l-d) - inductance.zeta(v, d) + 1/inductance.chi(v, l-d) - 1/inductance.chi(v, d)

def J_bulk_cross(p, x, l, d):
    a = x - p
    return (1/(2*np.pi)) * (_g(p, l, d) + inductance.zeta(a, l-d) - inductance.zeta(a, d))

def J_corner_cross(p, x, l, d):
    a = x - p
    termA = (1/(4*np.pi*d**2)) * (a**2/inductance.chi(a, d) - a*abs(a))
    termB = (p/(4*np.pi*d**2)) * (d/inductance.chi(d, p) - p)
    termC = (1/(4*np.pi)) * (inductance.zeta(p, l) - inductance.zeta(p, l-d) + inductance.zeta(a, l) - inductance.zeta(a, l-d))
    termD = (1/(4*np.pi)) * (1/inductance.chi(p, d) - 1 + 1/inductance.chi(p, l) - 1/inductance.chi(p, l-d))
    return termA + termB + termC + termD

def strip_avg_cross(x, l, d):
    left, _ = sciint.quad(lambda p: J_bulk_cross(p, x, l, d) + 2*J_corner_cross(p, x, l, d),
                           x-d+1e-13, x-1e-13, epsabs=1e-13, epsrel=1e-13, limit=200)
    def J_gt(p, x, l, d):
        a = x - p
        return (1/(2*np.pi)) * (1/inductance.chi(p, l) + inductance.zeta(p, l) + inductance.zeta(a, l) - 1
                                 - np.log((p-x)/p))
    right, _ = sciint.quad(J_gt, x+1e-13, x+d-1e-13, args=(x, l, d), epsabs=1e-13, epsrel=1e-13, limit=200)
    return (left + right) / (2*d)

def onaxis_half(x, l, d):
    return (inductance.self_inductance_dx(1.0, d, l, x)
            - inductance.self_inductance_correction_dx(1.0, d, x)) / 2

x, l, d = 50.0, 100.0, 1.0
sa = strip_avg_cross(x, l, d)
oh = onaxis_half(x, l, d)
full_dLdx = inductance.self_inductance_dx(1.0, d, l, x)
offset = sa - oh
pct_of_dLdx = offset / full_dLdx
print(f"  strip_avg_cross = {sa:.8f}")
print(f"  (1/2)(fully-linked dL/dx) [on-axis] = {oh:.8f}")
print(f"  offset = {offset:.8f}  =  {pct_of_dLdx:.3%} of dL/dx (paper: ~2.5% at l/d=100)")
report('B4 strip-force offset as % of dL/dx, l/d=100', f'{pct_of_dLdx:.3%}', '~2.5% (order of magnitude / sign)',
       0.01 < pct_of_dLdx < 0.05)

# self-force cancellation (bar exerts no net force on itself, by symmetry)
def B_bar(p, h, x, l, d):
    pm = x - p
    inside = (0.0 <= h <= l) and (abs(pm) < d)
    if inside:
        return (1/(4*np.pi*d**2)) * pm * (inductance.chi(l-h, pm) + inductance.chi(h, pm))
    return (1/(4*np.pi*pm)) * (inductance.chi(l-h, pm) + inductance.chi(h, pm))

max_asym = 0.0
for delta in np.linspace(1e-4, d-1e-4, 8):
    for h in np.linspace(1e-3, l-1e-3, 8):
        max_asym = max(max_asym, abs(B_bar(x+delta, h, x, l, d) + B_bar(x-delta, h, x, l, d)))
report('B4 bar self-force cancellation (odd in p about the bar axis)', f'{max_asym:.2e}', '~0 (exact, by symmetry)', max_asym < 1e-9)


  strip_avg_cross = 0.97646828
  (1/2)(fully-linked dL/dx) [on-axis] = 0.93266012
  offset = 0.04380817  =  2.453% of dL/dx (paper: ~2.5% at l/d=100)
[PASS] B4 strip-force offset as % of dL/dx, l/d=100: measured=2.453%  threshold=~2.5% (order of magnitude / sign)  
[PASS] B4 bar self-force cancellation (odd in p about the bar axis): measured=0.00e+00  threshold=~0 (exact, by symmetry)  


### B5. Second-order convergence and $\sim10^{-8}$ energy conservation

> "A dedicated convergence study, including the dissipationless limit
> $\rho=0$..., confirms second-order convergence
> ($|\varepsilon|\propto\Delta t^{2.0}$) and a worst-case transient
> relative energy error of $\sim10^{-8}$ at the time step used throughout
> ($\Delta t=2\times10^{-4}$)." (body, Section "Algorithm")

Runs the integrator at several step sizes and fits the observed
convergence order of the relative energy error at $\rho=0$ (where the
continuum equations conserve $\frac12Mv^2+\frac12LI^2$ exactly, isolating
the integrator's own error).

In [12]:
l, x0, B0 = 100.0, 500.0, 0.3
p0 = -2.e-2
T_total = 20.0
dt_list = [2e-3, 1e-3, 5e-4, 2.5e-4, 1.25e-4]
E0 = p0**2/2

max_errs = []
for dt in dt_list:
    n_steps = int(T_total/dt)
    n_print = max(1, n_steps//2000)
    x_, p_, I_, F_, K_, Eb_, Q_ = dynamics.verlet_integrate(x0, p0, l, B0, 0.0, dt, n_steps, n_print)
    Etot = np.array(K_) + np.array(Eb_)
    rel_err = np.abs((Etot - E0)/E0)
    max_errs.append(np.max(rel_err))
    print(f"  dt={dt:.2e}: max|rel. energy err|={max_errs[-1]:.3e}")

order = np.polyfit(np.log(dt_list), np.log(max_errs), 1)[0]
print(f"  fitted convergence order: {order:.3f}")
report('B5 convergence order', f'{order:.3f}', '~2.0', 1.8 < order < 2.2)

# production step size
dt_prod = 2.e-4
n_steps = int(T_total/dt_prod)
n_print = max(1, n_steps//2000)
x_, p_, I_, F_, K_, Eb_, Q_ = dynamics.verlet_integrate(x0, p0, l, B0, 0.0, dt_prod, n_steps, n_print)
Etot = np.array(K_) + np.array(Eb_)
rel_err_prod = np.max(np.abs((Etot - E0)/E0))
print(f"  at production dt=2e-4: max|rel. energy err|={rel_err_prod:.3e}  (paper: ~1e-8)")
report('B5 energy error at production dt=2e-4', f'{rel_err_prod:.2e}', 'O(1e-8)', rel_err_prod < 1e-6)


  dt=2.00e-03: max|rel. energy err|=1.017e-06
  dt=1.00e-03: max|rel. energy err|=2.544e-07
  dt=5.00e-04: max|rel. energy err|=6.359e-08
  dt=2.50e-04: max|rel. energy err|=1.590e-08


  dt=1.25e-04: max|rel. energy err|=3.976e-09
  fitted convergence order: 2.000
[PASS] B5 convergence order: measured=2.000  threshold=~2.0  
  at production dt=2e-4: max|rel. energy err|=1.018e-08  (paper: ~1e-8)
[PASS] B5 energy error at production dt=2e-4: measured=1.02e-08  threshold=O(1e-8)  


### B6. Force-model fit quality

> "the fit achieves $\text{R}^2>0.996$ and RMSE$<10^{-4}$." (body,
> Section "Results", using `scipy.optimize.curve_fit`)

In [13]:
l, x0, p0, mu, rho, dt = 100.0, 500.0, -1.e-2, 1.0, 1.0, 2.e-4
B_large = 0.3
n_steps, n_print = 1_000_000, 20

x_l, p_l, I_l, F_l, K_l, Eb_l, Q_l = dynamics.verlet_integrate(x0, p0, l, B_large, rho, dt, n_steps, n_print)
t_l = n_print*dt*np.arange(len(x_l))
F_l = np.array(F_l[:len(t_l)])
Ftot_l = F_l[:, 0] + F_l[:, 1]

alpha = models.alpha_damp(l, rho)
omega_d = models.omega_predicted(l, x0, B_large, rho)
rsq, rmse, params = models.fit_to_function_fix_decay_freq([1e-4, 0], alpha, omega_d, t_l, Ftot_l)
print(f"  R-squared = {rsq:.6f}   RMSE = {rmse:.6e}")
report('B6 force-fit R^2', f'{rsq:.4f}', '>0.996', rsq > 0.996)
report('B6 force-fit RMSE', f'{rmse:.2e}', '<1e-4', rmse < 1e-4)


  R-squared = 0.997626   RMSE = 3.731769e-05
[PASS] B6 force-fit R^2: measured=0.9976  threshold=>0.996  
[PASS] B6 force-fit RMSE: measured=3.73e-05  threshold=<1e-4  


### B7. Over-/under-damped crossover field $B_{0,c}(\rho)$

> "$B_{0,c}(\rho)=\frac{M^{1/2}L^{1/2}}{l}\gamma(\rho)$, the field at the
> critically damped point ($\omega_0=\gamma$)... the coefficient of
> determination drops sharply as $B_0$ crosses $B_{0,c}$ from above."
> (body, Section "Results")

Checks the defining property directly: at $B_0=B_{0,c}(\rho)$, the
damped-oscillator model's predicted frequency $\omega_d=\sqrt{\omega_0^2-
\gamma^2}$ should vanish (the critically-damped point), and should be
real and positive just above $B_{0,c}$. (The damping rate the paper
writes $\gamma$ is `alpha_damp` in `models.py`.)

In [14]:
l, x0 = 100.0, 500.0
for rho in [0.3, 0.6, 1.0]:
    B0c = models.B0_critical(l, x0, rho)
    omega_d_at = models.omega_predicted(l, x0, B0c, rho)
    omega_d_above = models.omega_predicted(l, x0, B0c*1.2, rho)
    omega_d_below = models.omega_predicted(l, x0, B0c*0.8, rho)
    print(f"  rho={rho}: B0c={B0c:.5f}  omega_d(B0c)={omega_d_at:.2e}  "
          f"omega_d(1.2*B0c)={omega_d_above:.5f}  omega_d(0.8*B0c)={omega_d_below:.5f}")

ok = (omega_d_at < 1e-8) and (omega_d_above > 0) and (omega_d_below == 0.0)
report('B7 B0c is exactly the omega_d=0 crossover', f'omega_d(B0c)={omega_d_at:.2e}', '~0', ok)


  rho=0.3: B0c=0.01748  omega_d(B0c)=0.00e+00  omega_d(1.2*B0c)=0.03898  omega_d(0.8*B0c)=0.00000
  rho=0.6: B0c=0.03496  omega_d(B0c)=0.00e+00  omega_d(1.2*B0c)=0.07796  omega_d(0.8*B0c)=0.00000
  rho=1.0: B0c=0.05826  omega_d(B0c)=0.00e+00  omega_d(1.2*B0c)=0.12993  omega_d(0.8*B0c)=0.00000
[PASS] B7 B0c is exactly the omega_d=0 crossover: measured=omega_d(B0c)=0.00e+00  threshold=~0  


## Summary

In [15]:
print(f"{len(PASS)} passed, {len(FAIL)} failed, out of {len(PASS)+len(FAIL)} checks.")
if FAIL:
    print("FAILED:")
    for c in FAIL:
        print(" -", c)
else:
    print("Every claim reproduced here matches the paper's stated threshold.")


22 passed, 0 failed, out of 22 checks.
Every claim reproduced here matches the paper's stated threshold.
